# Chapter 2: How LLMs Work

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Model Comparison Lab** that sends the same BA/QA prompts to different models and compares response quality, latency, and cost. Understand how model choice affects your workflow.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Familiarity with tokens and API basics


In [ ]:
# Setup
!pip install -q openai python-dotenv tiktoken

import os
import time
import json
from openai import OpenAI
from dotenv import load_dotenv
import tiktoken

load_dotenv()
client = OpenAI()

# Models to compare
MODELS = ['gpt-4o-mini', 'gpt-4o']

## 1. Understanding Tokenization

Before comparing models, let's understand how text is broken into tokens -- the fundamental unit LLMs process.


In [ ]:
# Explore tokenization with BA/QA content
enc = tiktoken.encoding_for_model('gpt-4o')

sample_texts = [
    'As a user, I want to reset my password so that I can regain access to my account.',
    'Given the user is on the login page, When they click Forgot Password, Then an email is sent.',
    'BRD-2024-Q3: The system shall support multi-factor authentication for all admin users.',
]

for text in sample_texts:
    tokens = enc.encode(text)
    print(f'Text: {text[:60]}...')
    print(f'  Tokens: {len(tokens)} | Characters: {len(text)} | Ratio: {len(text)/len(tokens):.1f} chars/token')
    print(f'  First 10 tokens decoded: {[enc.decode([t]) for t in tokens[:10]]}')
    print()

## 2. Model Comparison on BA/QA Tasks

We send identical prompts to different models and measure quality and performance.


In [ ]:
def compare_models(prompt: str, models: list) -> list:
    """Send the same prompt to multiple models and compare results."""
    results = []
    for model in models:
        start = time.time()
        response = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.3,
            max_tokens=500
        )
        elapsed = time.time() - start
        content = response.choices[0].message.content
        results.append({
            'model': model,
            'response': content,
            'latency_sec': round(elapsed, 2),
            'prompt_tokens': response.usage.prompt_tokens,
            'completion_tokens': response.usage.completion_tokens,
            'total_tokens': response.usage.total_tokens
        })
    return results

# BA task: Generate acceptance criteria
ba_prompt = (
    'Write acceptance criteria for this user story:\n'
    'As a bank customer, I want to transfer money between my accounts '
    'so that I can manage my finances easily.\n'
    'Use Given/When/Then format. Include at least 3 scenarios.'
)

results = compare_models(ba_prompt, MODELS)
for r in results:
    print(f"\n{'='*60}")
    print(f"Model: {r['model']} | Latency: {r['latency_sec']}s | Tokens: {r['total_tokens']}")
    print(f"{'='*60}")
    print(r['response'][:500])

In [ ]:
# QA task: Generate test scenarios
qa_prompt = (
    'Generate test scenarios for a login page with these requirements:\n'
    '- Username field (email format, max 100 chars)\n'
    '- Password field (min 8 chars, must include uppercase, lowercase, number)\n'
    '- Remember Me checkbox\n'
    '- Login button\n'
    '- Forgot Password link\n\n'
    'For each scenario, specify: ID, Description, Steps, Expected Result, Priority.'
)

qa_results = compare_models(qa_prompt, MODELS)
for r in qa_results:
    print(f"\nModel: {r['model']} | Tokens: {r['total_tokens']} | Latency: {r['latency_sec']}s")
    print(r['response'][:400])
    print('...')

## Exercises
1. Add a third model (e.g., `gpt-3.5-turbo`) and compare cost-effectiveness for simple tasks.
2. Vary the `temperature` parameter (0.0, 0.5, 1.0) and observe how outputs change for test case generation.
3. Measure how token count scales with prompt complexity -- try simple vs. detailed requirements.
4. Create a scoring rubric and use one model to evaluate another model's output.
